# 03 - Engenharia de Características Cinemáticas e Rotulagem de Manobras
**Unidades Curriculares:** Aprendizagem Automática (AAUTIA) & Sistemas de Decisão e Planeamento (PLNTDIA)

Neste notebook aplicamos transformações vetorizadas para extrair características dinâmicas no referencial do veículo ego:
- Velocidade de aproximação (*closing speed*) e *Time-To-Collision* (TTC).
- Ângulo de azimute e *gaps* longitudinais e laterais.
- Acelerações estimadas e taxa de variação de guinada (*yaw rate*).
- Rotulagem multiclasse de manobras táticas a horizonte de 3.0s (`STRAIGHT`, `TURN_LEFT`, `TURN_RIGHT`, `DECELERATING`, `ACCELERATING`).


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.features.pandas_feature_pipeline import PandasFeaturePipeline


## 1. Carregamento e Extração Cinemática Vetorizada


In [ ]:
df_samples = pd.read_csv(ROOT / "data" / "processed" / "predict_to_plan_samples.csv")
df_feat = PandasFeaturePipeline.add_vectorized_kinematic_features(df_samples)
print("Features cinemáticas calculadas com sucesso:")
print(list(df_feat[["closing_speed", "ttc", "azimuth_angle", "longitudinal_gap", "lateral_gap", "yaw_rate_1s", "agent_accel_est"]].head(3)))


## 2. Rotulagem Vetorizada de Manobras Táticas a 3.0s


In [ ]:
df_labeled = PandasFeaturePipeline.assign_maneuver_labels(df_feat)
maneuver_counts = df_labeled["maneuver_class"].value_counts()

plt.figure(figsize=(8, 4))
maneuver_counts.plot(kind="bar", color="#3b82f6", edgecolor="black")
plt.title("Distribuição das Classes de Manobras Táticas (Horizonte t+3.0s)")
plt.xlabel("Classe da Manobra")
plt.ylabel("Contagem de Amostras")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

print(maneuver_counts)


## 3. Contrato de Separação de Features (Garantia de Invariância Espacial)
Asseguramos que metadados e coordenadas globais são estritamente excluídos do vetor de treino $X$.


In [ ]:
X, y_reg, y_cls, groups, meta = PandasFeaturePipeline.get_feature_target_split(df_labeled)
print(f"Dimensão da Matriz de Entrada X: {X.shape}")
print(f"Dimensão dos Targets de Regressão y_reg: {y_reg.shape}")
print(f"Dimensão do Target de Classificação y_cls: {y_cls.shape}")
print("\nColunas em X:")
for c in X.columns:
    print(f"  • {c}")
